# Data Exploration — Air Quality & Low Emission Zone (LEZ) Paris

## Objective
First exploration of all data sources (Airparif NO₂, ZFE perimeters, INSEE IRIS income, weather)
to understand their structure and identify issues before cleaning and analysis.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from IPython.display import display

sys.path.insert(0, str(Path().resolve().parent))
from src.data_loader import load_airparif

print("Environment ready")


## 1. Airparif — NO₂ measurements

In [8]:
df_2019 = load_airparif(2019)
df_2020 = load_airparif(2020)
df_2021 = load_airparif(2021)
df_2022 = load_airparif(2022)
df_2023 = load_airparif(2023)

print("Data loaded ✓")

Data loaded ✓


In [15]:
for year, df in zip([2019, 2020, 2021, 2022, 2023], [df_2019, df_2020, df_2021, df_2022, df_2023]):
    print(f"{year}: {df.shape}")

2019 : (8760, 41)
2020 : (8784, 40)
2021 : (8760, 40)
2022 : (8760, 40)
2023 : (8760, 40)


**Observations:**
- Datetime index parsed correctly
- 2019: 8,760 records, 41 stations
- 2020: 8,784 records (leap year), 40 stations
- 2021–2023: 8,760 records, 40 stations
- One station disappears between 2019 and 2020 — to identify and assess before analysis

### Identifying the missing station

In [6]:
stations_2019 = set(df_2019.columns)
stations_2020 = set(df_2020.columns)

disappeared = stations_2019 - stations_2020
appeared = stations_2020 - stations_2019

print(f"Station(s) present in 2019 but not in 2020: {disappeared}")
print(f"Station(s) present in 2020 but not in 2019: {appeared}")

Station(s) present in 2019 but not in 2020: {'PA04C'}
Station(s) present in 2020 but not in 2019: set()


**Observation — Station PA04C (Paris Centre):**
- Present only in 2019, absent from 2020 to 2023
- No replacement station identified
- Decision: exclude PA04C from the entire analysis to ensure temporal consistency

### Missing values

In [ ]:
# Exclude PA04C upfront
dfs = {year: df.drop(columns=["PA04C"], errors="ignore")
       for year, df in zip([2019, 2020, 2021, 2022, 2023],
                           [df_2019, df_2020, df_2021, df_2022, df_2023])}

# Overall NaN rate per station (averaged across all 5 years)
nan_global = pd.concat(dfs.values()).isnull().mean() * 100
nan_global = nan_global.sort_values(ascending=False)

display(nan_global.round(1).to_frame("NaN %"))


In [ ]:
# NaN rate per station and per year
nan_per_year = pd.DataFrame({
    year: df.isnull().mean() * 100
    for year, df in dfs.items()
})

display(nan_per_year.round(1))


In [10]:
# Exclusion threshold: stations with more than 20% NaN over the full period
THRESHOLD = 20
excluded_stations = nan_global[nan_global > THRESHOLD].index.tolist()

print(f"Excluded stations (> {THRESHOLD}% NaN): {excluded_stations}")
print(f"Retained stations: {len(nan_global) - len(excluded_stations)} / {len(nan_global)}")

Excluded stations (> 20% NaN): ['DEF', 'BASCH', 'PA01H', 'SOULT']
Retained stations: 36 / 40


**Observations on Missing values:**
- 4 stations exceed the 20% NaN threshold and are excluded: DEF, BASCH, PA01H, SOULT
- The per-year breakdown reveals critical patterns:
  - DEF is 83.5% missing in 2021 and 100% in 2022 : absent during the core ZFE analysis window
  - BASCH is 100% missing in 2019 : no baseline data available
- AUT passes the global threshold (19.8%) but reaches 77.6% NaN in 2023 : flagged for monitoring
- Beyond 20%, imputation would introduce more noise than signal

- **36 stations retained out of 40**


### Outlier detection

In [ ]:
EXCLUDED = ["PA04C", "DEF", "BASCH", "PA01H", "SOULT"]

clean_dfs = {year: df.drop(columns=EXCLUDED, errors="ignore")
             for year, df in zip([2019, 2020, 2021, 2022, 2023],
                               [df_2019, df_2020, df_2021, df_2022, df_2023])}

all_data = pd.concat(clean_dfs.values())

# Negative values (physically impossible for NO2)
neg = (all_data < 0).sum()
print("Negative values per station:")
neg_pos = neg[neg > 0]
if len(neg_pos):
    display(neg_pos.to_frame("Negative count"))
else:
    print("None")

# Extreme values
print("Max value per station:")
display(all_data.max().sort_values(ascending=False).round(1).to_frame("Max NO2 (µg/m³)"))


**Observations on Outlier detection:**
- No systematic anomalies detected
- RUR-SO: 19 negative values — physically impossible, will be replaced by NaN in cleaning
- Max values (200–245 µg/m³) are physically plausible for high-traffic roadside stations during peak pollution episodes

In [14]:
# Datetime index consistency check
for year, df in clean_dfs.items():
    expected = pd.date_range(start=df.index.min(), end=df.index.max(), freq="h")
    missing_hours = expected.difference(df.index)
    duplicates = df.index.duplicated().sum()
    print(f"{year}: {len(missing_hours)} missing hours, {duplicates} duplicate timestamps")

2019: 0 missing hours, 0 duplicate timestamps
2020: 0 missing hours, 0 duplicate timestamps
2021: 0 missing hours, 0 duplicate timestamps
2022: 0 missing hours, 0 duplicate timestamps
2023: 0 missing hours, 0 duplicate timestamps


**Observations on Datetime index:**
- No missing hours and no duplicate timestamps across all 5 years
- The index is ready for time series analysis

## 2. ZFE perimeters

In [ ]:
zfe = gpd.read_file("../data/raw/zfe_perimetres/zone-a-faibles-emissions.geojson")

print(f"Shape: {zfe.shape}")
print("Column dtypes:")
print(zfe.dtypes)
display(zfe.head())


**Observations on ZFE perimeters:**
- Single perimeter covering the Paris metropolitan area
- Crit'Air 3 restriction applies to all vehicle types (vp_critair: V3)
- `date_debut` reflects the latest update (2025-01-01), not the original restriction date
- The Crit'Air 3 ban effective date (June 2021) will be hardcoded based on official sources
- `geometry` column contains the zone polygon — ready for spatial join with Airparif stations

## 3. INSEE IRIS — Median income

In [ ]:
insee = pd.read_csv(
    "../data/raw/insee_iris/BASE_TD_FILO_IRIS_2021_DISP.csv",
    sep=";",
    dtype={"IRIS": str, "DEP": str}
)

print(f"Shape: {insee.shape}")
display(insee.head())


In [6]:
# Filter on Paris and inner suburbs (departments 75, 92, 93, 94)
idf_depts = ["75", "92", "93", "94"]
insee_idf = insee[insee["IRIS"].str[:2].isin(idf_depts)]
print(f"Île-de-France IRIS: {len(insee_idf)} / {len(insee)}")

Île-de-France IRIS: 2745 / 16026


**Observations on INSEE IRIS:**
- 16,026 IRIS nationally — filtered to 2,745 for Paris and inner suburbs (depts 75, 92, 93, 94)
- Key column: `DISP_MED21` (median disposable income)
- All numeric columns stored as strings with comma decimal separator — to be converted in cleaning
- Department code extracted from first 2 digits of IRIS identifier

## 4. Weather data (Open-Meteo)

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 48.8566,
    "longitude": 2.3522,
    "start_date": "2019-01-01",
    "end_date": "2023-12-31",
    "hourly": "windspeed_10m,precipitation,temperature_2m",
    "timezone": "Europe/Paris"
}

response = requests.get(url, params=params)
meteo_raw = response.json()

meteo = pd.DataFrame(meteo_raw["hourly"])
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.set_index("time")

print(f"Shape: {meteo.shape}")
print(f"NaN total: {meteo.isnull().sum().sum()}")
display(meteo.head())


**Observations on Weather data (Open-Meteo):**
- 43,824 hourly records covering 2019–2023 (5 years × 8,760h, accounting for leap year 2020)
- 3 variables: wind speed (m/s), precipitation (mm), temperature (°C)
- No missing values
- Datetime index aligned with Airparif data — ready for merge in cleaning